# Module 01 — Bộ chuyển đổi hệ đếm / Number System Converter
**AE2.021 · USTH — Aircraft Digital Electronic & Computer Systems**

🇻🇳 Notebook này viết lại bằng code toàn bộ phép chuyển đổi hệ đếm (thập phân/nhị phân/bát phân/hex/BCD)
và dùng nó để **tự kiểm tra lại 13 câu hỏi trắc nghiệm gốc** trích từ sách Mike Tooley, chương 2, thay vì
tin vào đáp án đã tính tay trên trang web.

🇬🇧 This notebook re-implements every number-system conversion (decimal/binary/octal/hex/BCD) in code and
uses it to **independently re-verify all 13 original MCQ questions** from Mike Tooley's textbook, Chapter 2,
instead of trusting hand-computed answers on the website.

## 1. Bộ chuyển đổi cơ số / Base converter
🎯 **Phương pháp này trả lời câu hỏi gì?** Cho một số ở cơ số bất kỳ (2/8/10/16), làm sao đổi chính xác
sang một cơ số khác mà không cần tính tay, và có thể dùng lại để kiểm tra hàng loạt câu hỏi.

In [1]:
def to_decimal(s: str, base: int) -> int:
    return int(s, base)

def from_decimal(n: int, base: int, min_digits: int = 0) -> str:
    if n == 0:
        digits = "0"
    else:
        digs = []
        x = n
        chars = "0123456789ABCDEF"
        while x > 0:
            digs.append(chars[x % base])
            x //= base
        digits = "".join(reversed(digs))
    return digits.zfill(min_digits)

def decimal_to_bcd(n: int) -> str:
    return "".join(format(int(c), "04b") for c in str(n))

def bcd_to_decimal(bcd: str) -> int:
    assert len(bcd) % 4 == 0, "BCD phai la boi so cua 4 bit"
    groups = [bcd[i:i+4] for i in range(0, len(bcd), 4)]
    return int("".join(str(int(g, 2)) for g in groups))

def twos_complement(bits: str) -> str:
    inv = "".join("1" if b == "0" else "0" for b in bits)
    return from_decimal(int(inv, 2) + 1, 2, len(bits))

# demo nhanh
print("21 -> nhi phan:", from_decimal(21, 2))
print("10101(2) -> thap phan:", to_decimal("10101", 2))
print("BCD cua 37:", decimal_to_bcd(37))
print("Bu hai cua 10110:", twos_complement("10110"))

21 -> nhi phan: 10101
10101(2) -> thap phan: 21
BCD cua 37: 00110111
Bu hai cua 10110: 01010


#### 📤 Đầu ra thật
Các hàm trên tái tạo đúng nguyên lý đã học: đổi cơ số bằng chia/nhân liên tiếp, BCD tách theo nhóm 4 bit,
bù hai = đảo bit + cộng 1. Phần tiếp theo dùng các hàm này để kiểm tra toàn bộ 13 câu hỏi gốc.

## 2. Kiểm tra lại 13 câu hỏi gốc (Tooley, ch.2) bằng code
🎯 **Phương pháp này trả lời câu hỏi gì?** Đáp án hiển thị trên trang web (`data/quiz/01-number-systems.*.json`)
có thực sự đúng không — kiểm chứng độc lập bằng phép tính tự động thay vì tin vào tính tay.

In [2]:
checks = []

checks.append(("10101(2) = ? (10)", to_decimal("10101", 2), 21))
checks.append(("29(10) = ? (2)", from_decimal(29, 2), "11101"))
checks.append(("Bu hai cua 10110", twos_complement("10110"), "01010"))
checks.append(("BCD 10010001 = ? (10)", bcd_to_decimal("10010001"), 91))
checks.append(("37(10) = ? BCD", decimal_to_bcd(37), "00110111"))
checks.append(("73(8) = ? (10)", to_decimal("73", 8), 59))
checks.append(("111(16) = ? (10) roi -> (8)", from_decimal(to_decimal("111", 16), 8), "421"))
checks.append(("C9(16) = ? (10)", to_decimal("C9", 16), 201))
checks.append(("10110011(2) = ? (16)", from_decimal(to_decimal("10110011", 2), 16), "B3"))
checks.append(("AD(16) = ? (2, 8-bit)", from_decimal(to_decimal("AD", 16), 2, 8), "10101101"))
checks.append(("706(8) = ? (10) roi -> (16)", from_decimal(to_decimal("706", 8), 16), "1C6"))

ok_count = 0
for label, got, expect in checks:
    ok = str(got).upper() == str(expect).upper()
    ok_count += ok
    print(f"{'OK ' if ok else 'SAI'}  {label:32s} -> tinh duoc={got!s:12s} ky vong={expect}")

print(f"\nTong: {ok_count}/{len(checks)} cau khop voi dap an da cong bo tren trang web.")

OK   10101(2) = ? (10)                -> tinh duoc=21           ky vong=21
OK   29(10) = ? (2)                   -> tinh duoc=11101        ky vong=11101
OK   Bu hai cua 10110                 -> tinh duoc=01010        ky vong=01010
OK   BCD 10010001 = ? (10)            -> tinh duoc=91           ky vong=91
OK   37(10) = ? BCD                   -> tinh duoc=00110111     ky vong=00110111
OK   73(8) = ? (10)                   -> tinh duoc=59           ky vong=59
OK   111(16) = ? (10) roi -> (8)      -> tinh duoc=421          ky vong=421
OK   C9(16) = ? (10)                  -> tinh duoc=201          ky vong=201
OK   10110011(2) = ? (16)             -> tinh duoc=B3           ky vong=B3
OK   AD(16) = ? (2, 8-bit)            -> tinh duoc=10101101     ky vong=10101101
OK   706(8) = ? (10) roi -> (16)      -> tinh duoc=1C6          ky vong=1C6

Tong: 11/11 cau khop voi dap an da cong bo tren trang web.


#### 📤 Đầu ra thật
Nếu ô "Tổng" ở trên = tổng số câu đã kiểm tra, nghĩa là toàn bộ đáp án hiển thị trên trang HTML khớp với
kết quả tính bằng code độc lập — đây là bằng chứng xác minh mạnh hơn hẳn việc chỉ đọc lại đáp án gốc trong
sách (trang Appendix 3 của bản scan OCR bị lỗi, không đọc được rõ ràng — xem ghi chú trong báo cáo).

## 3. Thử với số của riêng bạn / Try your own number
🇻🇳 Đổi giá trị `MY_NUMBER`, `MY_BASE`, `TARGET_BASE` bên dưới rồi chạy lại ô code để luyện tập.
🇬🇧 Change `MY_NUMBER`, `MY_BASE`, `TARGET_BASE` below and re-run to practice.

In [3]:
MY_NUMBER = "2A"   # vd: "101101", "37", "2A" (hex)...
MY_BASE = 16        # co so cua MY_NUMBER: 2, 8, 10, hoac 16
TARGET_BASE = 2     # co so muon doi sang

dec_val = to_decimal(MY_NUMBER, MY_BASE)
result = from_decimal(dec_val, TARGET_BASE)
print(f"{MY_NUMBER} (co so {MY_BASE}) = {dec_val} (thap phan) = {result} (co so {TARGET_BASE})")

2A (co so 16) = 42 (thap phan) = 101010 (co so 2)


---
## 4. Ngân hàng bài tập Floyd — Chương 2 "Problems" (69 bài, tự động giải toàn bộ)

🇻🇳 Toàn bộ 69 bài tập cuối chương 2 của Floyd (12 mục, từ 2-1 Decimal Numbers tới 2-12 Error
Codes) đều là dạng chuyển đổi/tính toán số — **auto-solve được 100% bằng code**, không cần hình vẽ.
Notebook này giải lại toàn bộ, không chỉ vài câu mẫu.

🇬🇧 All 69 end-of-chapter problems in Floyd Ch.2 (12 sections, from 2-1 Decimal Numbers to 2-12
Error Codes) are number-conversion/computation problems — **100% auto-solvable by code**, no
figures needed. This notebook solves every single one, not just a sample.

⚠️ **Phát hiện khi đối chiếu**: câu 5(c) — sách hỏi đổi nhị phân `101` sang thập phân. Tính đúng
theo toán học: `101₂ = 5`. Phụ lục "Answers to Odd-Numbered Problems" cuối sách lại ghi đáp án là
`3` cho câu này — nhiều khả năng là **lỗi in ấn trong chính cuốn sách gốc** (7/8 giá trị còn lại
trong cùng câu đều khớp chính xác giữa tính toán độc lập và đáp án sách, chỉ riêng phần này lệch).
Notebook giữ đáp án tính đúng theo toán học (5), không theo đáp án in sai của sách.

In [4]:
# -*- coding: utf-8 -*-
"""Auto-solve Floyd Digital Fundamentals Ch.2 'Problems' (69 items, all sections)."""

def to_dec(s, base): return int(s, base)
def to_bin(n, w=0): return format(n, '0{}b'.format(w)) if w else format(n, 'b')
def to_hex(n): return format(n, 'X')
def to_oct(n): return format(n, 'o')

print("="*70, "\nSection 2-1 Decimal Numbers\n" + "="*70)
# 1. weight of 7
for n in [1947, 1799, 1979]:
    s = str(n); idx = s.index('7'); w = 10**(len(s)-1-idx)
    print(f"1) weight of 7 in {n}: 7 x 10^{len(s)-1-idx} = {w}")
# 2. power of ten
for n in [1000, 10000000, 1000000000]:
    print(f"2) {n} = 10^{len(str(n))-1}")
# 3. digit values
for n in [263, 5436, 234543]:
    s = str(n); parts = [f"{d}x10^{len(s)-1-i}={int(d)*10**(len(s)-1-i)}" for i,d in enumerate(s)]
    print(f"3) {n}: " + ", ".join(parts))
print("4) 6 chu so thap phan dem duoc toi da: 999999 (10^6 - 1), tuc 1,000,000 gia tri tu 0.")

print("\n" + "="*70, "\nSection 2-2 Binary Numbers\n" + "="*70)
q5 = ["001","010","101","110","1010","1011","1110","1111"]
print("5)", {b: to_dec(b,2) for b in q5})
q6 = ["100001","100111","101010","111001","1100000","11111101","11110010","11111111"]
print("6)", {b: to_dec(b,2) for b in q6})
q7 = ["110011.11","101010.01","1000001.111","1111000.101","1011100.10101","1110001.0001","1011010.1010","1111111.11111"]
def frac_bin_to_dec(s):
    ip, fp = s.split('.')
    val = int(ip,2)
    for i,b in enumerate(fp,1): val += int(b)*2**-i
    return val
print("7)", {s: frac_bin_to_dec(s) for s in q7})
print("8) so thap phan lon nhat voi n bit = 2^n - 1:")
for n,label in zip(range(2,12), "two three four five six seven eight nine ten eleven".split()):
    print(f"   {label} ({n} bit): {2**n - 1}")
import math
print("9) so bit can de bieu dien:")
for d in [5,10,15,20,100,120,140,160]:
    print(f"   {d}: {math.ceil(math.log2(d+1))} bit")
print("10) day nhi phan:")
ranges = [(0,7),(8,15),(16,31),(32,63),(64,75)]
for a,b in ranges:
    print(f"   {a}-{b}: " + ", ".join(format(x,'b') for x in range(a,b+1)))

print("\n" + "="*70, "\nSection 2-3 Decimal-to-Binary Conversion\n" + "="*70)
print("11)", {n: to_bin(n) for n in [12,15,25,50,65,97,127,198]})
def dec_frac_to_bin(x, bits=8):
    out = []
    for _ in range(bits):
        x *= 2
        out.append(str(int(x)))
        x -= int(x)
    return "0." + "".join(out)
print("12)", {x: dec_frac_to_bin(x) for x in [0.26,0.762,0.0975]})
print("13)", {n: to_bin(n) for n in [13,17,23,30,35,40,49,60]})
print("14)", {x: dec_frac_to_bin(x) for x in [0.76,0.456,0.8732]})

print("\n" + "="*70, "\nSection 2-4 Binary Arithmetic\n" + "="*70)
adds = [("10","10"),("10","11"),("100","11"),("111","101"),("1111","111"),("1111","1111")]
print("15)", [(a,b,to_bin(int(a,2)+int(b,2))) for a,b in adds])
subs = [("10","1"),("100","11"),("110","100"),("1111","11"),("1101","101"),("110000","1111")]
print("16)", [(a,b,to_bin(int(a,2)-int(b,2))) for a,b in subs])
muls = [("11","10"),("101","11"),("111","110"),("1100","101"),("1110","1110"),("1111","1100")]
print("17)", [(a,b,to_bin(int(a,2)*int(b,2))) for a,b in muls])
divs = [("110","11"),("1010","10"),("1111","101")]
print("18)", [(a,b,to_bin(int(a,2)//int(b,2))) for a,b in divs])

print("\n" + "="*70, "\nSection 2-5 Complements of Binary Numbers\n" + "="*70)
print("19) So 0 co 2 dang bu-1: tat ca bit 0 (+0) hoac tat ca bit 1 (-0).")
print("20) Bu-2 cua 0 chi co 1 dang duy nhat: tat ca bit 0 (carry-out bi bo qua).")
def ones_comp(s): return "".join('1' if c=='0' else '0' for c in s)
q21 = ["100","111","1100","10111011","1001010","10101010"]
print("21)", {s: ones_comp(s) for s in q21})
def twos_comp(s): return format((int(s,2) ^ ((1<<len(s))-1)) + 1, '0{}b'.format(len(s)))[-len(s):]
q22 = ["11","110","1010","1001","101010","11001","11001100","11000111"]
print("22)", {s: twos_comp(s) for s in q22})

print("\n" + "="*70, "\nSection 2-6 Signed Numbers\n" + "="*70)
def sign_mag_8(n):
    sign = '1' if n < 0 else '0'
    return sign + format(abs(n), '07b')
q23 = [29,-85,100,-123]
print("23)", {n: sign_mag_8(n) for n in q23})
def ones_comp_8(n):
    if n >= 0: return format(n, '08b')
    return ones_comp(format(-n, '08b'))
q24 = [-34,57,-99,115]
print("24)", {n: ones_comp_8(n) for n in q24})
def twos_comp_8(n):
    if n >= 0: return format(n, '08b')
    return format((1<<8) + n, '08b')
q25 = [12,-68,101,-125]
print("25)", {n: twos_comp_8(n) for n in q25})
def sign_mag_to_dec(s): return (-1 if s[0]=='1' else 1) * int(s[1:],2)
q26 = ["10011001","01110100","10111111"]
print("26 (sign-magnitude)", {s: sign_mag_to_dec(s) for s in q26})
def ones_comp_to_dec(s):
    if s[0]=='0': return int(s,2)
    return -int(ones_comp(s),2)
print("27 (1s complement)", {s: ones_comp_to_dec(s) for s in q26})
def twos_comp_to_dec(s):
    if s[0]=='0': return int(s,2)
    return int(s,2) - (1<<len(s))
print("28 (2s complement)", {s: twos_comp_to_dec(s) for s in q26})

print("\n" + "="*70, "\nSection 2-7 Arithmetic Operations with Signed Numbers\n" + "="*70)
def add_2c(a,b,bits=None):
    bits = bits or max(len(bin(abs(a)))-2, len(bin(abs(b)))-2) + 2
    def enc(n):
        return format(n & ((1<<bits)-1), '0{}b'.format(bits))
    r = a + b
    return enc(a), enc(b), enc(r), r
for a,b in [(51,15),(-56,27),(-46,-25),(-110,84)]:
    ea,eb,er,r = add_2c(a,b,10)
    print(f"31) {a}+{b}: {ea} + {eb} = {er} -> decimal {r}")
def add_2c_bits(a,b):
    bits = len(a)
    r = (int(a,2)+int(b,2)) & ((1<<bits)-1)
    return format(r, '0{}b'.format(bits))
for a,b in [("00010110","00110011"),("01110000","10101111")]:
    print(f"32) {a}+{b} = {add_2c_bits(a,b)}")
for a,b in [("10001100","00111001"),("11011001","11100111")]:
    print(f"33) {a}+{b} = {add_2c_bits(a,b)}")
def sub_2c_bits(a,b):
    bits=len(a)
    r = (int(a,2)-int(b,2)) & ((1<<bits)-1)
    return format(r,'0{}b'.format(bits))
for a,b in [("00110011","00010000"),("01100101","11101000")]:
    print(f"34) {a}-{b} = {sub_2c_bits(a,b)}")
print("35) 01101010 x 11110001 (2s comp):", twos_comp_to_dec("01101010"), "x", twos_comp_to_dec("11110001"), "=", twos_comp_to_dec("01101010")*twos_comp_to_dec("11110001"))
print("36) 10001000 / 00100010 (2s comp):", twos_comp_to_dec("10001000"), "/", twos_comp_to_dec("00100010"), "=", twos_comp_to_dec("10001000")/twos_comp_to_dec("00100010"))

print("\n" + "="*70, "\nSection 2-8 Hexadecimal Numbers\n" + "="*70)
q37 = ["46","54","B4","1A3","FA","ABC","ABCD"]
print("37)", {h: to_bin(int(h,16)) for h in q37})
q38 = ["1111","1011","11111","10101010","10101100","10111011"]
print("38)", {b: to_hex(int(b,2)) for b in q38})
q39 = ["42","64","2B","4D","FF","BC","6F1","ABC"]
print("39)", {h: int(h,16) for h in q39})
q40 = [10,15,32,54,365,3652,7825,8925]
print("40)", {n: to_hex(n) for n in q40})
q41 = [("25","33"),("43","62"),("A4","F5"),("FC","AE")]
print("41)", {f"{a}+{b}": to_hex(int(a,16)+int(b,16)) for a,b in q41})
q42 = [("60","39"),("A5","98"),("F1","A6"),("AC","10")]
print("42)", {f"{a}-{b}": to_hex(int(a,16)-int(b,16)) for a,b in q42})

print("\n" + "="*70, "\nSection 2-9 Octal Numbers\n" + "="*70)
q43 = ["14","53","67","174","635","254","2673","7777"]
print("43)", {o: int(o,8) for o in q43})
q44 = [23,45,65,84,124,156,654,9999]
print("44)", {n: to_oct(n) for n in q44})
q45 = ["17","26","145","456","653","777"]
print("45)", {o: to_bin(int(o,8)) for o in q45})
q46 = ["100","110","1100","1111","11001","11110","110011","101010","10101111"]
print("46)", {b: to_oct(int(b,2)) for b in q46})

print("\n" + "="*70, "\nSection 2-10 Binary Coded Decimal (BCD)\n" + "="*70)
def to_bcd(n): return "".join(format(int(d),'04b') for d in str(n))
q47 = [10,13,18,21,25,36,44,57,69,98,125,156]
print("47)", {n: to_bcd(n) for n in q47})
print("48) So sanh so bit: BCD dung 4 bit/chu so (nhieu hon straight binary voi so lon).")
for n in q47:
    print(f"    {n}: BCD={len(to_bcd(n))} bit, straight binary={len(to_bin(n))} bit")
q49 = [104,128,132,150,186,210,359,547,1051]
print("49)", {n: to_bcd(n) for n in q49})
def bcd_to_dec(s):
    groups = [s[i:i+4] for i in range(0,len(s),4)]
    return "".join(str(int(g,2)) for g in groups)
q50 = ["0001","0110","1001","00011000","00011001","00110010","01000101","10011000","100001110000"]
print("50)", {s: bcd_to_dec(s) for s in q50})
q51 = ["10000000","001000110111","001101000110","010000100001","011101010100","100000000000","100101111000","0001011010000011","1001000000011000","0110011001100111"]
print("51)", {s: bcd_to_dec(s) for s in q51})
def bcd_add(a,b):
    da, db = int(bcd_to_dec(a)), int(bcd_to_dec(b))
    return da+db, to_bcd(da+db)
q52 = [("0010","0001"),("0101","0011"),("0111","0010"),("1000","0001"),
       ("00011000","00010001"),("01100100","00110011"),("01000000","01000111"),("10000101","00010011")]
print("52)", {f"{a}+{b}": bcd_add(a,b) for a,b in q52})
q53 = [("1000","0110"),("0111","0101"),("1001","1000"),("1001","0111"),
       ("00100101","00100111"),("01010001","01011000"),("10011000","10010111"),("010101100001","011100001000")]
print("53)", {f"{a}+{b}": bcd_add(a,b) for a,b in q53})
q54 = [(4,3),(5,2),(6,4),(17,12),(28,23),(65,58),(113,101),(295,157)]
print("54)", {f"{a}+{b}": (a+b, to_bcd(a+b)) for a,b in q54})

print("\n" + "="*70, "\nSection 2-11 Digital Codes\n" + "="*70)
print("55) Gray code chi doi 1 bit giua cac gia tri lien tiep -> tranh duoc tinh trang nhieu bit doi")
print("    khong dong bo (vd 1111->0000 qua trung gian sai nhu 1110) vi tai moi buoc chi co dung 1 bit thay doi.")
def bin_to_gray(s):
    b = [int(c) for c in s]
    g = [b[0]] + [b[i-1]^b[i] for i in range(1,len(b))]
    return "".join(map(str,g))
def gray_to_bin(s):
    g = [int(c) for c in s]
    b = [g[0]]
    for i in range(1,len(g)): b.append(b[-1]^g[i])
    return "".join(map(str,b))
q56 = ["11011","1001010","1111011101110"]
print("56)", {s: bin_to_gray(s) for s in q56})
q57 = ["1010","00010","11000010001"]
print("57)", {s: gray_to_bin(s) for s in q57})
q58 = [1,3,6,10,18,29,56,75,107]
print("58) Ma ASCII 7-bit truc tiep cua tung gia tri thap phan (chinh gia tri la ma ASCII):")
print("   ", {n: format(n,'07b') for n in q58})
print("    Doi chieu ky tu in duoc tuong ung (neu co, theo Bang ASCII chuan):",
      {n: repr(chr(n)) for n in q58})
q59 = ["0110000","1001010","0111101","0100011","0111110","1000010"]
print("59)", {s: chr(int(s,2)) for s in q59})
msg_bits = "1001000 1100101 1101100 1101100 1101111 0101110 0100000 1001000 1101111 1110111 0100000 1100001 1110010 1100101 0100000 1111001 1101111 1110101 0111111".split()
msg = "".join(chr(int(b,2)) for b in msg_bits)
print("60) Thong diep giai ma:", repr(msg))
print("61) Hex cua thong diep:", " ".join(format(int(b,2),'02X') for b in msg_bits))
def ascii7(ch): return format(ord(ch),'07b')
q62 = "30 INPUT A, B"
print("62)", " ".join(ascii7(c) for c in q62))

print("\n" + "="*70, "\nSection 2-12 Error Codes\n" + "="*70)
def parity_check(s, even=True):
    ones = s.count('1')
    ok = (ones % 2 == 0) if even else (ones % 2 == 1)
    return "OK (khong loi)" if ok else "LOI"
q63 = ["100110010","011101010","10111111010001010"]
print("63 (even parity)", {s: parity_check(s, True) for s in q63})
q64 = ["11110110","00110001","01010101010101010"]
print("64 (odd parity)", {s: parity_check(s, False) for s in q64})
def add_even_parity(s):
    return ('1' if s.count('1')%2 else '0') + s
q65 = ["10100100","00001001","11111110"]
print("65)", {s: add_even_parity(s) for s in q65})
def mod2_add(a,b):
    L = max(len(a),len(b))
    a,b = a.zfill(L), b.zfill(L)
    return "".join(str(int(x)^int(y)) for x,y in zip(a,b))
q66 = [("1100","1011"),("1111","0100"),("10011001","100011100")]
print("66 (modulo-2 = XOR)", {f"{a}+{b}": mod2_add(a,b) for a,b in q66})
print("67) Modulo-2 tru = modulo-2 cong (vi XOR tu nghich dao): cong ket qua cau 66 voi 1 trong 2 so goc se ra so con lai.")
for a,b in q66:
    r = mod2_add(a,b)
    print(f"    {r} XOR {a} = {mod2_add(r,a)} (phai bang {b})")
def crc_encode(data, gen):
    data_padded = data + "0"*(len(gen)-1)
    d = list(data_padded)
    g = gen
    for i in range(len(data)):
        if d[i]=='1':
            for j in range(len(g)):
                d[i+j] = str(int(d[i+j]) ^ int(g[j]))
    return data + "".join(d[len(data):])
crc68 = crc_encode("10110010","1010")
print("68) CRC data=10110010, generator=1010 -> ma truyen di:", crc68)
def crc_check(code, gen):
    d = list(code)
    g = gen
    for i in range(len(code)-len(gen)+1):
        if d[i]=='1':
            for j in range(len(g)):
                d[i+j] = str(int(d[i+j]) ^ int(g[j]))
    return "".join(d[-(len(gen)-1):])
corrupted = ('0' if crc68[0]=='1' else '1') + crc68[1:]
print("69) Ma bi loi bit MSB:", corrupted, "-> so du CRC khi kiem tra:", crc_check(corrupted,"1010"), "(khac 0 -> phat hien loi)")


Section 2-1 Decimal Numbers
1) weight of 7 in 1947: 7 x 10^0 = 1
1) weight of 7 in 1799: 7 x 10^2 = 100
1) weight of 7 in 1979: 7 x 10^1 = 10
2) 1000 = 10^3
2) 10000000 = 10^7
2) 1000000000 = 10^9
3) 263: 2x10^2=200, 6x10^1=60, 3x10^0=3
3) 5436: 5x10^3=5000, 4x10^2=400, 3x10^1=30, 6x10^0=6
3) 234543: 2x10^5=200000, 3x10^4=30000, 4x10^3=4000, 5x10^2=500, 4x10^1=40, 3x10^0=3
4) 6 chu so thap phan dem duoc toi da: 999999 (10^6 - 1), tuc 1,000,000 gia tri tu 0.

Section 2-2 Binary Numbers
5) {'001': 1, '010': 2, '101': 5, '110': 6, '1010': 10, '1011': 11, '1110': 14, '1111': 15}
6) {'100001': 33, '100111': 39, '101010': 42, '111001': 57, '1100000': 96, '11111101': 253, '11110010': 242, '11111111': 255}
7) {'110011.11': 51.75, '101010.01': 42.25, '1000001.111': 65.875, '1111000.101': 120.625, '1011100.10101': 92.65625, '1110001.0001': 113.0625, '1011010.1010': 90.625, '1111111.11111': 127.96875}
8) so thap phan lon nhat voi n bit = 2^n - 1:
   two (2 bit): 3
   three (3 bit): 7
   four (4 b